<a href="https://colab.research.google.com/github/Fu-Pei-Yin/Deep-Generative-Mode/blob/week5/%E4%BD%BF%E7%94%A8_Seq2Seq%E7%94%9F%E6%88%90%E3%80%8C%E6%9C%AA%E4%BE%86%E5%AD%B8%E7%BF%92%E8%A1%8C%E7%82%BA%E5%BA%8F%E5%88%97%E3%80%8D%E6%95%B8%E6%93%9A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# 設置隨機種子
torch.manual_seed(42)
np.random.seed(42)

# 檢查GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 上傳資料文件
print("請上傳以下文件：studentInfo.csv, studentVle.csv, studentAssessment.csv")
print("請依次上傳這三個文件...")

# 這裡使用上傳功能
from google.colab import files
uploaded = files.upload()

# 讀取資料
student_info = pd.read_csv('studentInfo.csv')
student_vle = pd.read_csv('studentVle.csv')
student_assessment = pd.read_csv('studentAssessment.csv')

print("資料讀取完成!")
print(f"studentInfo shape: {student_info.shape}")
print(f"studentVle shape: {student_vle.shape}")
print(f"studentAssessment shape: {student_assessment.shape}")

# 資料預處理函數
def prepare_weekly_sequences(student_vle, student_assessment, student_info):
    """準備每週序列資料"""

    # 合併點擊資料
    weekly_clicks = student_vle.groupby(['id_student', 'date']).agg({
        'sum_click': 'sum'
    }).reset_index()

    # 合併作業提交資料
    weekly_assignments = student_assessment.groupby(['id_student', 'date_submitted']).agg({
        'score': ['mean', 'count']
    }).reset_index()
    weekly_assignments.columns = ['id_student', 'date', 'avg_score', 'submit_count']

    # 建立完整的學生列表
    all_students = student_info['id_student'].unique()

    sequences = []

    for student in all_students:
        # 獲取該學生的點擊資料
        student_clicks = weekly_clicks[weekly_clicks['id_student'] == student].copy()
        student_assignments = weekly_assignments[weekly_assignments['id_student'] == student].copy()

        if len(student_clicks) < 6:  # 至少需要6週資料（4週輸入+2週輸出）
            continue

        # 按日期排序
        student_clicks = student_clicks.sort_values('date')
        student_assignments = student_assignments.sort_values('date')

        # 建立每週特徵
        weeks = sorted(student_clicks['date'].unique())

        for i in range(len(weeks) - 5):
            # 過去4週作為輸入，未來2週作為輸出
            input_weeks = weeks[i:i+4]
            output_weeks = weeks[i+4:i+6]

            # 提取輸入特徵
            input_features = []
            for week in input_weeks:
                week_data = {}

                # 點擊數
                click_data = student_clicks[student_clicks['date'] == week]
                week_data['clicks'] = click_data['sum_click'].sum() if not click_data.empty else 0

                # 作業提交
                assign_data = student_assignments[student_assignments['date'] == week]
                week_data['submit_cnt'] = assign_data['submit_count'].sum() if not assign_data.empty else 0
                week_data['has_submit'] = 1 if week_data['submit_cnt'] > 0 else 0

                # 累積平均分數（到該週為止）
                past_assignments = student_assignments[student_assignments['date'] <= week]
                week_data['avg_score_sofar'] = past_assignments['avg_score'].mean() if not past_assignments.empty else 0

                input_features.append(week_data)

            # 提取輸出（未來2週的點擊數）
            output_clicks = []
            for week in output_weeks:
                click_data = student_clicks[student_clicks['date'] == week]
                output_clicks.append(click_data['sum_click'].sum() if not click_data.empty else 0)

            # 計算點擊數差分
            clicks_values = [f['clicks'] for f in input_features]
            clicks_diff = [clicks_values[j+1] - clicks_values[j] for j in range(len(clicks_values)-1)]
            clicks_diff.append(0)  # 最後一週沒有差分

            # 添加差分到特徵中
            for j, feature in enumerate(input_features):
                feature['clicks_diff1'] = clicks_diff[j]

            sequences.append({
                'student_id': student,
                'input_features': input_features,
                'output_clicks': output_clicks
            })

    return sequences

print("開始處理序列資料...")
sequences = prepare_weekly_sequences(student_vle, student_assessment, student_info)
print(f"總序列數: {len(sequences)}")

# 資料集類別
class StudentSequenceDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences

        # 提取所有特徵進行標準化
        all_inputs = []
        all_outputs = []

        for seq in sequences:
            # 輸入特徵
            input_vec = []
            for week_feat in seq['input_features']:
                input_vec.extend([
                    week_feat['clicks'],
                    week_feat['submit_cnt'],
                    week_feat['has_submit'],
                    week_feat['avg_score_sofar'],
                    week_feat['clicks_diff1']
                ])
            all_inputs.append(input_vec)

            # 輸出
            all_outputs.append(seq['output_clicks'])

        # 標準化
        self.scaler_input = StandardScaler()
        self.scaler_output = StandardScaler()

        self.inputs = self.scaler_input.fit_transform(all_inputs)
        self.outputs = self.scaler_output.fit_transform(all_outputs)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        input_tensor = torch.FloatTensor(self.inputs[idx]).view(4, 5)  # 4週, 5個特徵
        output_tensor = torch.FloatTensor(self.outputs[idx])  # 2週點擊數
        return input_tensor, output_tensor

# 分割資料集
student_ids = list(set([seq['student_id'] for seq in sequences]))
train_ids, temp_ids = train_test_split(student_ids, test_size=0.3, random_state=42)
valid_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=42)

train_sequences = [seq for seq in sequences if seq['student_id'] in train_ids]
valid_sequences = [seq for seq in sequences if seq['student_id'] in valid_ids]
test_sequences = [seq for seq in sequences if seq['student_id'] in test_ids]

print(f"訓練集: {len(train_sequences)} 序列")
print(f"驗證集: {len(valid_sequences)} 序列")
print(f"測試集: {len(test_sequences)} 序列")

# 建立資料加載器
train_dataset = StudentSequenceDataset(train_sequences)
valid_dataset = StudentSequenceDataset(valid_sequences)
test_dataset = StudentSequenceDataset(test_sequences)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# Seq2Seq LSTM 模型
class Seq2SeqLSTM(nn.Module):
    def __init__(self, input_dim=5, hidden_dim=64, output_dim=2, num_layers=1):
        super(Seq2SeqLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # Encoder
        self.encoder_lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)

        # Decoder
        self.decoder_lstm = nn.LSTM(hidden_dim, hidden_dim, num_layers, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # Encoder
        _, (hidden, cell) = self.encoder_lstm(x)

        # Decoder輸入（使用encoder的最後狀態）
        decoder_input = torch.zeros(x.size(0), 2, self.hidden_dim).to(x.device)

        # Decoder
        decoder_output, _ = self.decoder_lstm(decoder_input, (hidden, cell))
        output = self.fc_out(decoder_output)

        return output

# Seq2Seq VAE 模型
class Seq2SeqVAE(nn.Module):
    def __init__(self, input_dim=5, hidden_dim=64, latent_dim=16, output_dim=2, num_layers=1):
        super(Seq2SeqVAE, self).__init__()
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim
        self.num_layers = num_layers

        # Encoder
        self.encoder_lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        # Decoder
        self.decoder_lstm = nn.LSTM(latent_dim, hidden_dim, num_layers, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def encode(self, x):
        _, (hidden, _) = self.encoder_lstm(x)
        hidden_last = hidden[-1]  # 取最後一層的最後狀態
        mu = self.fc_mu(hidden_last)
        logvar = self.fc_logvar(hidden_last)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        # 重複z作為decoder輸入
        decoder_input = z.unsqueeze(1).repeat(1, 2, 1)
        decoder_output, _ = self.decoder_lstm(decoder_input)
        output = self.fc_out(decoder_output)
        return output

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        reconstructed = self.decode(z)
        return reconstructed, mu, logvar

# 訓練函數
def train_lstm(model, train_loader, valid_loader, epochs=10):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()

    train_losses = []
    valid_losses = []

    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # 驗證
        model.eval()
        valid_loss = 0
        with torch.no_grad():
            for data, target in valid_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                valid_loss += criterion(output, target).item()

        train_loss /= len(train_loader)
        valid_loss /= len(valid_loader)
        train_losses.append(train_loss)
        valid_losses.append(valid_loss)

        if epoch % 5 == 0:
            print(f'Epoch {epoch}: Train Loss: {train_loss:.4f}, Valid Loss: {valid_loss:.4f}')

    return train_losses, valid_losses

def train_vae(model, train_loader, valid_loader, epochs=10, beta=0.1):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    train_losses = []
    valid_losses = []

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        train_recon = 0
        train_kld = 0

        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            reconstructed, mu, logvar = model(data)

            # 計算損失
            recon_loss = nn.MSELoss()(reconstructed, target)
            kld_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
            kld_loss /= (data.size(0) * 2)  # 正則化

            loss = recon_loss + beta * kld_loss

            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_recon += recon_loss.item()
            train_kld += kld_loss.item()

        # 驗證
        model.eval()
        valid_loss = 0
        with torch.no_grad():
            for data, target in valid_loader:
                data, target = data.to(device), target.to(device)
                reconstructed, mu, logvar = model(data)

                recon_loss = nn.MSELoss()(reconstructed, target)
                kld_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
                kld_loss /= (data.size(0) * 2)

                loss = recon_loss + beta * kld_loss
                valid_loss += loss.item()

        train_loss /= len(train_loader)
        valid_loss /= len(valid_loader)
        train_losses.append(train_loss)
        valid_losses.append(valid_loss)

        if epoch % 5 == 0:
            print(f'Epoch {epoch}: Train Loss: {train_loss:.4f}, Valid Loss: {valid_loss:.4f}')
            print(f'  Recon: {train_recon/len(train_loader):.4f}, KLD: {train_kld/len(train_loader):.4f}')

    return train_losses, valid_losses

# 快速訓練（減少epochs以加快速度）
print("開始訓練 LSTM 模型...")
lstm_model = Seq2SeqLSTM(input_dim=5, hidden_dim=64, output_dim=2)
lstm_train_loss, lstm_valid_loss = train_lstm(lstm_model, train_loader, valid_loader, epochs=5)

print("\n開始訓練 VAE 模型...")
vae_model = Seq2SeqVAE(input_dim=5, hidden_dim=64, latent_dim=16, output_dim=2)
vae_train_loss, vae_valid_loss = train_vae(vae_model, train_loader, valid_loader, epochs=5, beta=0.1)

# 評估函數
def evaluate_models(lstm_model, vae_model, test_loader, n_samples=20):
    lstm_model.eval()
    vae_model.eval()

    all_lstm_preds = []
    all_vae_samples = []
    all_targets = []

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)

            # LSTM 預測
            lstm_pred = lstm_model(data)
            all_lstm_preds.append(lstm_pred.cpu())

            # VAE 多樣本生成
            batch_vae_samples = []
            for _ in range(n_samples):
                vae_reconstructed, _, _ = vae_model(data)
                batch_vae_samples.append(vae_reconstructed.cpu())

            all_vae_samples.append(torch.stack(batch_vae_samples, dim=1))
            all_targets.append(target.cpu())

    # 合併所有批次
    lstm_preds = torch.cat(all_lstm_preds, dim=0)
    vae_samples = torch.cat(all_vae_samples, dim=0)
    targets = torch.cat(all_targets, dim=0)

    return lstm_preds, vae_samples, targets

# 反標準化
def inverse_transform_output(scaled_data):
    return test_dataset.scaler_output.inverse_transform(scaled_data)

# 評估模型
print("開始評估模型...")
lstm_preds, vae_samples, targets = evaluate_models(lstm_model, vae_model, test_loader, n_samples=20)

# 反標準化回原始尺度
lstm_preds_original = inverse_transform_output(lstm_preds.numpy())
vae_samples_original = np.stack([inverse_transform_output(vae_samples[:, i, :].numpy())
                               for i in range(vae_samples.shape[1])], axis=1)
targets_original = inverse_transform_output(targets.numpy())

# 計算評估指標
def calculate_metrics(lstm_preds, vae_samples, targets):
    # LSTM MSE
    lstm_mse = np.mean((lstm_preds - targets) ** 2)

    # VAE Best-of-N MSE
    vae_best_mse = np.min(np.mean((vae_samples - targets[:, np.newaxis, :]) ** 2, axis=(0,2)), axis=0)

    # Diversity (std)
    diversity = np.std(vae_samples, axis=1).mean()

    # Coverage (比例)
    threshold = np.median(np.mean((lstm_preds - targets) ** 2, axis=1))
    coverage = np.mean([
        np.any(np.mean((vae_samples[i] - targets[i]) ** 2, axis=1) <= threshold)
        for i in range(len(targets))
    ])

    return lstm_mse, vae_best_mse, diversity, coverage, threshold

lstm_mse, vae_best_mse, diversity, coverage, threshold = calculate_metrics(
    lstm_preds_original, vae_samples_original, targets_original
)

print("\n=== 評估結果 (原始尺度) ===")
print(f"LSTM MSE: {lstm_mse:.4f}")
print(f"VAE Best-of-N MSE: {vae_best_mse:.4f}")
print(f"Diversity (std): {diversity:.4f}")
print(f"Coverage (比例): {coverage:.4f}")
print(f"門檻 tau = LSTM 每序列 MSE 中位數 = {threshold:.4f}")

# 計算改進分佈
def calculate_improvement_distribution(lstm_preds, vae_samples, targets):
    lstm_sequence_mse = np.mean((lstm_preds - targets) ** 2, axis=1)
    vae_best_sequence_mse = np.min(np.mean((vae_samples - targets[:, np.newaxis, :]) ** 2, axis=2), axis=1)

    improvement = lstm_sequence_mse - vae_best_sequence_mse

    # 定義改進區間
    buckets = [
        ('VAE<<劣(>1000)', -10000, -1000),
        ('VAE劣(200~1000)', -1000, -200),
        ('VAE劣(50~200)', -200, -50),
        ('VAE劣(10~50)', -50, -10),
        ('VAE略劣(<10)', -10, 0),
        ('~打平(±10)', -10, 10),
        ('VAE略勝(10~50)', 10, 50),
        ('VAE勝(50~200)', 50, 200),
        ('VAE大勝(200~1000)', 200, 1000),
        ('VAE>>大勝(>1000)', 1000, 10000)
    ]

    distribution = []
    for label, low, high in buckets:
        count = np.sum((improvement >= low) & (improvement < high))
        ratio = count / len(improvement)
        distribution.append((label, count, ratio))

    return distribution, improvement

improvement_dist, improvement_values = calculate_improvement_distribution(
    lstm_preds_original, vae_samples_original, targets_original
)

print("\n=== Improvement by bucket (Δ = LSTM MSE - VAE best MSE) ===")
for label, count, ratio in improvement_dist:
    print(f"{label}: {count:6d} {ratio:.4f}")

# 找出最差的5個案例
def find_top_regressed(lstm_preds, vae_samples, targets, improvement_values, n=5):
    # 找出VAE表現最差的案例（improvement最小）
    worst_indices = np.argsort(improvement_values)[:n]

    results = []
    for i, idx in enumerate(worst_indices):
        lstm_mse_seq = np.mean((lstm_preds[idx] - targets[idx]) ** 2)
        vae_best_mse_seq = np.min(np.mean((vae_samples[idx] - targets[idx]) ** 2, axis=1))
        improvement = lstm_mse_seq - vae_best_mse_seq

        diversity_std = np.std(vae_samples[idx], axis=0).mean()

        results.append({
            'idx': idx,
            'LSTM_MSE': lstm_mse_seq,
            'VAE_best_MSE': vae_best_mse_seq,
            'improvement': improvement,
            'y_true': targets[idx],
            'y_LSTM': lstm_preds[idx],
            'y_VAE_best': vae_samples[idx][np.argmin(np.mean((vae_samples[idx] - targets[idx]) ** 2, axis=1))],
            'Diversity_std': diversity_std
        })

    return results

top_regressed = find_top_regressed(
    lstm_preds_original, vae_samples_original, targets_original, improvement_values, n=5
)

print("\n=== Top-5 Regressed (VAE best >> LSTM) ===")
for result in top_regressed:
    print(f"idx: {result['idx']}")
    print(f"  LSTM_MSE: {result['LSTM_MSE']:.2f}, VAE_best_MSE: {result['VAE_best_MSE']:.2f}")
    print(f"  Improvement: {result['improvement']:.2f}")
    print(f"  y_true: {result['y_true']}")
    print(f"  y_LSTM: {[f'{x:.2f}' for x in result['y_LSTM']]}")
    print(f"  y_VAE_best: {[f'{x:.2f}' for x in result['y_VAE_best']]}")
    print(f"  Diversity_std: {result['Diversity_std']:.4f}")
    print()

# 視覺化結果
def plot_comparison(lstm_preds, vae_samples, targets, n_examples=3):
    fig, axes = plt.subplots(1, n_examples, figsize=(15, 5))

    if n_examples == 1:
        axes = [axes]

    for i in range(n_examples):
        idx = np.random.randint(len(targets))

        # 真實值
        axes[i].plot([1, 2], targets[idx], 'go-', linewidth=3, markersize=8, label='Ground Truth')

        # LSTM預測
        axes[i].plot([1, 2], lstm_preds[idx], 'ro-', linewidth=2, markersize=6, label='LSTM')

        # VAE多樣本
        for j in range(vae_samples.shape[1]):
            axes[i].plot([1, 2], vae_samples[idx, j], 'b-', alpha=0.3, linewidth=1)

        # VAE平均
        vae_mean = np.mean(vae_samples[idx], axis=0)
        axes[i].plot([1, 2], vae_mean, 'bo-', linewidth=2, markersize=6, label='VAE Mean')

        axes[i].set_title(f'Example {i+1}')
        axes[i].set_xlabel('Week')
        axes[i].set_ylabel('Clicks')
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

print("\n視覺化比較結果:")
plot_comparison(lstm_preds_original, vae_samples_original, targets_original, n_examples=3)

# 輸出最終結果表格
print("\n" + "="*50)
print("最終評估結果總結")
print("="*50)
print(f"{'Metric':<20} {'Value':<15}")
print(f"{'LSTM MSE':<20} {lstm_mse:.4f}")
print(f"{'VAE Best-of-N MSE':<20} {vae_best_mse:.4f}")
print(f"{'Diversity (std)':<20} {diversity:.4f}")
print(f"{'Coverage (比例)':<20} {coverage:.4f}")

print("\n改進分佈:")
for label, count, ratio in improvement_dist:
    print(f"{label:<20} {count:>6d} ({ratio:.4f})")

Using device: cpu
請上傳以下文件：studentInfo.csv, studentVle.csv, studentAssessment.csv
請依次上傳這三個文件...
